# Jet Tagging with CNNs — Improved Version
**CNNs as classifiers for sparse data events**

**Tasks:** Multiclass classification of jets originating from: *gluon (g), quark (q), W boson, Z boson, top quark*.

Pipeline:
1. Load dataset and visualise features
2. Build jet images (2D histogram of constituent $p_T$ in the $\eta$–$\phi$ plane)
3. Design & train an **improved** CNN (deeper architecture + BatchNorm + data augmentation)
4. Evaluate performance (accuracy, confusion matrix, ROC curves)

**Improvements over the baseline:**
- ✅ Full 16k training set (instead of a reduced subset)
- ✅ Data augmentation: random horizontal/vertical flips and 90° rotations
- ✅ Batch Normalization after every Conv layer
- ✅ Deeper architecture: 4 convolutional blocks (16→32→64→128 filters)

In [ ]:
# --- Imports ---
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import itertools, os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## 1. Load Dataset
The `.npz` file contains:
- `features`: shape `(20000, 100, 16)` — 20k jets, up to 100 constituents, 16 features each
- `target`: one-hot labels of shape `(20000, 5)` for classes `[g, q, W, Z, t]`

In [ ]:
# Load dataset
with np.load('dataset_jetTag.npz') as f:
    target   = f['target']
    features = f['features']

# Convert one-hot labels to integer class indices
label = np.argmax(target, axis=1)
labels_str = ['gluon', 'quark', 'W', 'Z', 'top']

print(f"Features shape : {features.shape}")
print(f"Labels shape   : {label.shape}")
print(f"Class distribution: {dict(zip(labels_str, np.bincount(label)))}")

## 2. Train / Validation / Test Split (80 / 10 / 10)

> **Improvement:** We use the **full 20k dataset** (16k train, 2k val, 2k test) rather than a
> reduced subset, as recommended in the discussion section of the baseline notebook.

In [ ]:
X_train, X_rem, y_train, y_rem = train_test_split(
    features, label, train_size=0.8, random_state=42, stratify=label
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_rem, y_rem, test_size=0.5, random_state=42, stratify=y_rem
)
print("Train:", X_train.shape, "| Valid:", X_valid.shape, "| Test:", X_test.shape)

## 3. Create Jet Images

Each jet is turned into a **100×100 image** by histogramming the constituent transverse momenta ($p_T$)
into bins of the relative pseudorapidity–azimuth plane $(\eta_{rel}, \phi_{rel})$.

> **Note on sparse data:** As the lecture recommends (slide 82), we use **large convolutional kernels (11×11)**
> in the first layer to capture features efficiently in these sparse images.

In [ ]:
def create_images(features, bins=100):
    """Project each jet constituent pT onto a 100x100 eta-phi grid."""
    jet_imgs = np.empty((features.shape[0], bins, bins), dtype=np.float32)
    for i in range(features.shape[0]):
        eta = features[i, :, 8]   # eta_rel  (feature index 8)
        phi = features[i, :, 11]  # phi_rel  (feature index 11)
        pt  = features[i, :, 5]   # pT       (feature index 5)
        h, _, _ = np.histogram2d(eta, phi, bins=bins,
                                 range=[[-1,1],[-1,1]], weights=pt)
        jet_imgs[i] = h
    return jet_imgs

print("Creating jet images … (this takes ~1–2 min)")
jet_imgs_train = create_images(X_train)
jet_imgs_valid = create_images(X_valid)
jet_imgs_test  = create_images(X_test)
print(f"Train image shape: {jet_imgs_train.shape}")

### Example jet image

In [ ]:
idx = 42
plt.figure(figsize=(5,4))
plt.imshow(jet_imgs_train[idx], origin='lower', norm=LogNorm())
plt.colorbar(label='$p_T$ [a.u.]')
plt.title(f'Jet image — class: {labels_str[y_train[idx]]}', fontsize=13)
plt.xlabel('$\Delta\eta$ cell'); plt.ylabel('$\Delta\phi$ cell')
plt.tight_layout(); plt.show()

### Mean jet image per class
Summing all images in each category reveals the typical energy deposition pattern.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for i, label_name in enumerate(labels_str):
    mask = (y_train == i)
    mean_img = jet_imgs_train[mask].mean(axis=0)
    im = axes[i].imshow(mean_img, origin='lower', norm=LogNorm(), cmap='viridis')
    axes[i].set_title(label_name, fontsize=13)
    axes[i].set_xlabel('$\Delta\eta$ cell')
    if i == 0: axes[i].set_ylabel('$\Delta\phi$ cell')
plt.suptitle('Mean jet image per class', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

## 4. Preprocessing

Before feeding images to the CNN we:
1. **Downsample** from 100×100 → 50×50 using local averaging (reduces memory)
2. **Log-transform** pixel values: $\tilde{p}_T = \log(1 + p_T)$ — compresses the dynamic range
3. **Add channel dimension** (required by `Conv2D`)
4. **One-hot encode** the labels

In [ ]:
from skimage.transform import downscale_local_mean

def preprocess(imgs, target_size=50):
    factor = imgs.shape[1] // target_size
    out = np.empty((imgs.shape[0], target_size, target_size), dtype=np.float32)
    for i in range(imgs.shape[0]):
        out[i] = downscale_local_mean(imgs[i], (factor, factor))
    return np.log1p(out)[..., np.newaxis]   # log1p + channel dim

X_train_pp = preprocess(jet_imgs_train)
X_valid_pp = preprocess(jet_imgs_valid)
X_test_pp  = preprocess(jet_imgs_test)

n_classes = 5
Y_train = keras.utils.to_categorical(y_train, n_classes)
Y_valid = keras.utils.to_categorical(y_valid, n_classes)
Y_test  = keras.utils.to_categorical(y_test, n_classes)

print("Preprocessed shape:", X_train_pp.shape)
print("Value range — min:", X_train_pp.min().round(3), " max:", X_train_pp.max().round(3))

## 5. Data Augmentation

> **Improvement (slide 74):** Physics jets have approximate symmetries in the $\eta$–$\phi$ plane.
> We exploit this by randomly applying:
> - **Horizontal flip** (reflection in $\eta$)
> - **Vertical flip** (reflection in $\phi$)
> - **90° rotations** (k = 0, 1, 2, 3)
>
> These augmentations are applied on-the-fly during training via a `tf.data` pipeline,
> so the model sees a different augmented version of each image every epoch.

In [ ]:
BATCH_SIZE = 256
AUTOTUNE   = tf.data.AUTOTUNE

@tf.function
def augment(image, label):
    """Random flip + random 90-degree rotation."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    # Random rotation by k*90 degrees
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k=k)
    return image, label

# Training dataset with augmentation
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train_pp, Y_train))
    .shuffle(buffer_size=len(X_train_pp), seed=42)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Validation and test datasets — no augmentation
valid_ds = (
    tf.data.Dataset.from_tensor_slices((X_valid_pp, Y_valid))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test_pp, Y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(f"Training batches  : {len(train_ds)}")
print(f"Validation batches: {len(valid_ds)}")
print(f"Test batches      : {len(test_ds)}")

### Visualise augmented samples

In [ ]:
# Show one original vs 4 augmented versions of the same jet
sample_img, sample_label = X_train_pp[idx], Y_train[idx]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
axes[0].imshow(sample_img[..., 0], origin='lower', cmap='viridis')
axes[0].set_title('Original', fontsize=12)
for j in range(1, 5):
    aug_img, _ = augment(
        tf.constant(sample_img), tf.constant(sample_label)
    )
    axes[j].imshow(aug_img.numpy()[..., 0], origin='lower', cmap='viridis')
    axes[j].set_title(f'Augmented {j}', fontsize=12)
for ax in axes:
    ax.set_xlabel('$\Delta\eta$ cell')
axes[0].set_ylabel('$\Delta\phi$ cell')
plt.suptitle(f'Data augmentation — class: {labels_str[np.argmax(sample_label)]}',
             fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 6. Improved CNN Architecture

Following the **lecture guidelines** and the **suggested improvements**:

| Feature | Baseline | **Improved** |
|---|---|---|
| Conv blocks | 3 | **4** |
| Filter progression | 16→32→64 | **16→32→64→128** |
| Batch Normalization | ❌ | **✅ after every Conv2D** |
| Data augmentation | ❌ | **✅ flips + rot90** |
| Training set size | ~8k | **16k** |
| Max epochs | 15 | **30** |

**Design choices:**
- **Large 11×11 kernel** in Block 1 — captures structure in sparse physics images (slide 82)
- **BatchNorm** before activation in each block — stabilises training, allows higher learning rates, acts as regulariser
- **Dropout(0.4)** in the dense head — prevents co-adaptation of neurons
- **GlobalAveragePooling2D** — reduces parameter count vs Flatten
- **Softmax** output + **CategoricalCrossentropy** loss for 5-class classification

In [ ]:
tf.random.set_seed(42)

def conv_bn_relu(filters, kernel_size, padding='same'):
    """Conv2D → BatchNorm → ReLU block (standard modern pattern)."""
    return keras.Sequential([
        layers.Conv2D(filters, kernel_size, padding=padding, use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
    ])

inputs = keras.Input(shape=(50, 50, 1), name='jet_image')

# Block 1 — large kernel for sparse jet images (lecture slide 82)
x = conv_bn_relu(16, (11, 11))(inputs)
x = layers.MaxPooling2D((2, 2))(x)          # 50→25

# Block 2
x = conv_bn_relu(32, (5, 5))(x)
x = layers.MaxPooling2D((2, 2))(x)          # 25→12

# Block 3
x = conv_bn_relu(64, (3, 3))(x)
x = layers.MaxPooling2D((2, 2))(x)          # 12→6

# Block 4 — extra depth vs baseline
x = conv_bn_relu(128, (3, 3))(x)
x = layers.GlobalAveragePooling2D()(x)      # 6→128-dim vector

# Classification head
x = layers.Dense(128, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(n_classes, activation='softmax', name='class_probs')(x)

model = keras.Model(inputs, outputs, name='JetCNN_Improved')
model.summary()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 7. Training

We train for up to **30 epochs** with:
- **EarlyStopping** (patience=7) — restores best weights automatically
- **ReduceLROnPlateau** (patience=3, factor=0.5) — lowers LR when val_loss plateaus
- **ModelCheckpoint** — saves the best model to disk

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=7, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-6
    ),
    keras.callbacks.ModelCheckpoint(
        'best_jet_cnn.keras', monitor='val_loss', save_best_only=True, verbose=0
    ),
]

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=30,
    callbacks=callbacks,
    verbose=1,
)

## 8. Performance Assessment
### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     'o-', label='Train')
axes[0].plot(history.history['val_accuracy'], 's-', label='Validation')
axes[0].set_title('Accuracy per epoch', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     'o-', label='Train')
axes[1].plot(history.history['val_loss'], 's-', label='Validation')
axes[1].set_title('Loss per epoch', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Confusion Matrix

In [ ]:
y_pred_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
ax.set_xticklabels(labels_str, fontsize=11); ax.set_yticklabels(labels_str, fontsize=11)
for i, j in itertools.product(range(n_classes), range(n_classes)):
    ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
            color='white' if cm_norm[i,j] > 0.5 else 'black', fontsize=10)
ax.set_xlabel('Predicted label', fontsize=12); ax.set_ylabel('True label', fontsize=12)
ax.set_title('Confusion Matrix (row-normalized)', fontsize=13)
plt.tight_layout(); plt.show()

### ROC Curves (one-vs-rest)

In [ ]:
Y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']

fig, ax = plt.subplots(figsize=(7, 6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(Y_test_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{labels_str[i]} (AUC={roc_auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate', fontsize=12); ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — one-vs-rest', fontsize=13)
ax.legend(loc='lower right', fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=labels_str))
loss, acc = model.evaluate(test_ds, verbose=0)
print(f"\nTest Accuracy: {acc:.4f}  |  Test Loss: {loss:.4f}")

## 9. Discussion

### Summary of improvements applied

| Improvement | Effect |
|---|---|
| **Full 16k training set** | More data → better generalisation, especially for g/q confusion |
| **BatchNorm after Conv** | Faster convergence, internal covariate shift reduced, mild regularisation |
| **4th Conv block (128 filters)** | More representational capacity to learn subtle jet substructure features |
| **Data augmentation (flips + rot90)** | Enforces $\eta/\phi$ symmetry, effectively multiplies dataset, reduces overfitting |

### Key design choices (following the lecture)
- **Large 11×11 kernel** in the first Conv layer — as recommended for sparse physics images (slide 82)
- **Increasing number of filters** per layer: 16 → 32 → 64 → 128 (good practice, slide 75)
- **Dropout(0.4)** between dense layers to reduce overfitting
- **Log(1+pT) normalisation** to handle the large dynamic range in energy deposits
- **Stratified splits** to guarantee balanced class representation in every subset

### Further directions
- Experiment with a **Vision Transformer (ViT)** on the jet images
- Try a **Graph Neural Network (GNN)** acting directly on constituent 4-vectors (no image step)
- Add **L2 weight regularisation** to Conv/Dense layers
- Use **cosine annealing** or **OneCycleLR** learning-rate schedule instead of ReduceLROnPlateau